# Python Notebook 1: Data Understanding and Preprocessing
**Author:** Mihiri Pabasara | **Student ID:** [Your Student ID]
**Module:** 5DATA002W.2 – Machine Learning & Data Mining
**Peer Reviewer:** [Peer Reviewer Full Name] | **Review Date:** [Date of Review]

> All code reused from **Seminar 1 (Parts A–C)**, **Seminar 2 (Parts A–D)**, and **Code Reuse Session 1** as noted per cell.

**Dataset Column Mapping to Coursework Data Dictionary:**

| Dataset Column | CW Variable | Notes |
|---|---|---|
| id | ID | Unique identifier – dropped (no predictive value) |
| age | Age | Client age in years |
| income | Income | Annual gross income |
| home_ownership | Home Ownership | OWN/RENT/MORTGAGE/OTHER |
| emplyment_length | Employment Length | Years employed (source typo retained) |
| loan_intent | Loan Intent | Purpose of loan |
| loan_amount | Loan Amount | Requested loan value (GBP) |
| loan_interest_rate | Loan Interest Rate | % cost of borrowing |
| loan_income_ratio | Loan-to-Income Ratio (LTI) | loan_amount / income |
| payment_default_on_file | Payment Default on File | Y/N |
| credit_history_length | Credit History Length | Years of credit history |
| loan_approval_status | **Loan Approval Status (Target A)** | 0=Approved, 1=Rejected |
| max_allowed_loan | **Maximum Loan Amount (Target B)** | Max lending value (GBP) |

> **Note:** Sex, Education Qualifications, and Credit Application Acceptance are absent from the dataset per GDPR anonymisation stated in the coursework specification.

## 1. Import Libraries
**Reused From:** Seminar 1, Part (C) – importing pandas and related libraries

In [ ]:
# Import pandas for data manipulation
import pandas as pd
# Import numpy for numerical operations
import numpy as np
# Import plotly express for interactive visualisation
import plotly.express as px
# Import sklearn preprocessing for label encoding
from sklearn import preprocessing
# Import warnings to suppress non-critical messages
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded successfully.")

## 2. Mount Google Drive
**Reused From:** Seminar 1, Part (A) – mounting Google Drive in Colab

In [ ]:
from google.colab import drive
# Mount Google Drive to access the dataset
drive.mount('/content/drive')

## 3. Load Dataset
**Reused From:** Seminar 1, Part (C), Step 1 – pd.read_csv()

In [ ]:
# Load the loan approval dataset using pd.read_csv
df = pd.read_csv('/content/drive/MyDrive/loan_approval_data.csv')
# Display the first five rows of the dataset
df.head()

## 4. Dataset Dimensions
**Reused From:** Seminar 1, Part (C), Step 6 – shape method

In [ ]:
# Display the number of rows (instances) and columns (features)
print("Dataset shape (rows, columns):", df.shape)

## 5. Column Names
**Reused From:** Seminar 1, Part (C), Step 3 – list(df.columns)

In [ ]:
# Print all column names in the dataset
print("Columns:", list(df.columns))

## 6. Data Types and Info
**Reused From:** Seminar 1, Part (C), Step 4 – df.info()

In [ ]:
# Print dataset information including data types and non-null counts
df.info()

## 7. Descriptive Statistics
**Reused From:** Seminar 1, Part (C), Step 5 – df.describe()

In [ ]:
# Set display format to avoid scientific notation
pd.set_option('display.float_format', '{:.2f}'.format)
# Descriptive statistics for numeric variables
df.describe().transpose()

In [ ]:
# Descriptive statistics for categorical (object) variables
df.describe(include='object').transpose()

## 8. Missing Data Analysis
**Reused From:** Seminar 1, Part (C), Step 7 – isnull().sum() and isna().sum()/len()

In [ ]:
# Count missing values per variable
print("Missing value counts:")
print(df.isnull().sum())
print()
# Calculate percentage of missing values
print("Percentage of missing values:")
print(df.isna().sum() / len(df) * 100)

## 9. Target Variable Distribution
**Reused From:** Seminar 2, Part (A) – Univariate bar plot with plotly express

**Target variable encoding:**
- **0 = Approved**
- **1 = Rejected**

In [ ]:
#Target Variable Distribution

# Map numeric target to readable labels for visualisation
df_plot = df.copy()
df_plot['Approval Status'] = df_plot['loan_approval_status'].map({0: 'Approved (0)', 1: 'Rejected (1)'})

# Plot frequency distribution of loan approval status
fig = px.histogram(
    df_plot,
    x='Approval Status',
    color='Approval Status',
    title='Loan Approval Status Distribution',
    color_discrete_map={
        'Approved (0)': 'steelblue',
        'Rejected (1)': 'firebrick'
    }
)
# Update axis labels
fig.update_layout(yaxis_title='Count')
# Show figure
fig.show()

# Print counts and proportions
print(df['loan_approval_status'].value_counts())
print(df['loan_approval_status'].value_counts(normalize=True) * 100)

## 10. Outlier Visualisation
**Reused From:** Seminar 2, Part (B) – box plots for outlier detection

In [ ]:
# Box plot for Age to identify outliers
px.box(df, x='age', title='Box Plot: Age').show()
# Box plot for Employment Length to detect extreme values
px.box(df, x='emplyment_length', title='Box Plot: Employment Length').show()
# Box plot for Income to identify extreme outliers
px.box(df, x='income', title='Box Plot: Income').show()
# Box plot for Maximum Loan Amount to check for impossible values
px.box(df, x='max_allowed_loan', title='Box Plot: Maximum Loan Amount').show()

In [ ]:
# IQR-based outlier detection function (Seminar 2, Part B)
def find_outliers_IQR(series):
    # Calculate the 25th percentile (Q1)
    q1 = series.quantile(0.25)
    # Calculate the 75th percentile (Q3)
    q3 = series.quantile(0.75)
    # Compute the Interquartile Range
    IQR = q3 - q1
    # Identify points outside 1.5*IQR from Q1 and Q3
    return series[((series < (q1 - 1.5*IQR)) | (series > (q3 + 1.5*IQR)))]

# Apply to Age - check for impossible biological values
age_out = find_outliers_IQR(df['age'])
print(f"Age outliers ({len(age_out)}): max age in data = {df['age'].max()}")
# Apply to employment length - check for unrealistic values
emp_out = find_outliers_IQR(df['emplyment_length'])
print(f"Employment outliers ({len(emp_out)}): max = {df['emplyment_length'].max()}")
# Check for negative max_allowed_loan (impossible loan amounts)
print(f"Negative max_allowed_loan count: {(df['max_allowed_loan'] < 0).sum()}")

## 11. Data Cleaning
**Reused From:** Seminar 2, Part (B) – removing erroneous rows with drop(); Seminar 2, Part (C) – missing value imputation

### Issues Found and Proposed Fixes:

| # | Variable | Issue | Fix | Justification |
|---|---|---|---|---|
| 1 | age | 1 impossible value (age=123); 6 missing values | Remove outlier row; impute missing with median | Age of 123 is biologically impossible. Median is robust to skew. |
| 2 | emplyment_length | 3 extreme outliers >50 years | Remove rows exceeding 50 years | More than 50 years of employment is unrealistic for a loan applicant. |
| 3 | max_allowed_loan | 3 negative values | Drop erroneous rows | A negative loan amount is financially impossible. |
| 4 | loan_interest_rate | 11 missing values (<0.02%) | Impute with mean | Small proportion; mean is appropriate for continuous, near-normal data. |
| 5 | payment_default_on_file | 5 missing values; stored as string Y/N | Impute with mode; encode Y=1, N=0 | Mode preserves majority class; binary encoding needed for modelling. |
| 6 | home_ownership | Categorical string | Label encode | Machine learning algorithms require numeric inputs. |
| 7 | loan_intent | Categorical string | Label encode | Same reason as above. |

In [ ]:
# --- FIX 1a: Remove impossible age value (age=123 is biologically impossible) ---
print("Shape before cleaning:", df.shape)
# Drop any row where age exceeds 100 (impossible human age)
df = df[df['age'] <= 100]
print("Shape after removing age > 100:", df.shape)

# --- FIX 3: Remove negative max_allowed_loan values (impossible loan amounts) ---
# A loan amount cannot be negative; these are data entry errors
df = df[df['max_allowed_loan'] >= 0]
print("Shape after removing negative max_allowed_loan:", df.shape)

# --- FIX 2: Remove extreme employment length outliers (> 50 years unrealistic) ---
# Employment beyond 50 years is unrealistic for a loan applicant
df = df[df['emplyment_length'] <= 50]
print("Shape after removing emplyment_length > 50:", df.shape)

In [ ]:
# --- FIX 1b: Impute missing Age values with median (robust to skew) ---
med_age = df['age'].median()
print(f"Missing age before imputation: {df['age'].isnull().sum()}")
# Fill missing age values with the median
df['age'].fillna(med_age, inplace=True)
print(f"Missing age after imputation:  {df['age'].isnull().sum()}")

# --- FIX 4: Impute missing loan_interest_rate with mean ---
mean_rate = df['loan_interest_rate'].mean()
print(f"Missing loan_interest_rate before: {df['loan_interest_rate'].isnull().sum()}")
# Fill missing interest rate with the column mean
df['loan_interest_rate'].fillna(mean_rate, inplace=True)
print(f"Missing loan_interest_rate after:  {df['loan_interest_rate'].isnull().sum()}")

# --- FIX 5a: Impute missing payment_default_on_file with mode ---
mode_def = df['payment_default_on_file'].mode()[0]
print(f"Missing payment_default_on_file before: {df['payment_default_on_file'].isnull().sum()}")
# Fill missing default values with the most frequent category
df['payment_default_on_file'].fillna(mode_def, inplace=True)
print(f"Missing payment_default_on_file after:  {df['payment_default_on_file'].isnull().sum()}")

# Confirm all missing values have been resolved
print("\nAll missing values after cleaning:")
print(df.isnull().sum())

In [ ]:
# --- FIX 5b: Encode payment_default_on_file: Y=1 (default on file), N=0 (no default) ---
# Map string values to binary integers for modelling
df['payment_default_on_file'] = df['payment_default_on_file'].map({'Y': 1, 'N': 0})
print("payment_default_on_file unique values:", df['payment_default_on_file'].unique())

# --- FIX 6: Label encode home_ownership (categorical -> numeric) ---
le = preprocessing.LabelEncoder()
# Fit and transform home_ownership column
df['home_ownership'] = le.fit_transform(df['home_ownership'])
# Display the encoding mapping for reference
print("home_ownership encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# --- FIX 7: Label encode loan_intent (categorical -> numeric) ---
# Fit and transform loan_intent column
df['loan_intent'] = le.fit_transform(df['loan_intent'])
print("loan_intent unique encoded values:", sorted(df['loan_intent'].unique()))

# Confirm all data types are now numeric after encoding
print("\nData types after encoding:")
print(df.dtypes)

## 12. Verify Cleaned Dataset
**Reused From:** Seminar 1, Part (C) – describe() and shape to confirm clean data

In [ ]:
# Verify cleaned dataset shape and summary statistics
print("Cleaned dataset shape:", df.shape)
# Display descriptive statistics for the clean dataset
df.describe().transpose()

## 13. Create and Save Classification Dataset
**Reused From:** Seminar 6, Part 1 – separating datasets for classification

**Variables RETAINED for Classification:**
- All columns except `id` (unique identifier – no predictive value) and `max_allowed_loan` (regression target – including it would cause data leakage into the classification model)

**Variables DROPPED for Classification:**
- `id`: Unique row identifier with no predictive information
- `max_allowed_loan`: This is the regression target; it is only available for approved clients, so using it as a feature would cause data leakage

In [ ]:
# Drop id (no predictive value) and max_allowed_loan (regression target - data leakage risk)
df_class = df.drop(['id', 'max_allowed_loan'], axis=1)
# Display classification dataset shape
print("Classification dataset shape:", df_class.shape)
# Display feature names
print("Features:", list(df_class.columns))
# Preview first rows
df_class.head()

In [ ]:
# Save the classification dataset to CSV for use in Notebook 2
df_class.to_csv('/content/drive/MyDrive/loan_classification_data.csv', index=False)
print("Saved: loan_classification_data.csv")

## 14. Create and Save Regression Dataset
**Reused From:** Seminar 6, Part 1 – filtering approved applicants for regression

**Important:** In this dataset, `loan_approval_status = 0` means **Approved**.
Only approved clients have a `max_allowed_loan` value; rejected clients do not receive a maximum loan offer, so the regression dataset is filtered to approved clients only.

In [ ]:
# Filter only loan-APPROVED clients (loan_approval_status == 0 means Approved)
df_reg = df[df['loan_approval_status'] == 0].copy()
# Drop id (unnecessary identifier) and loan_approval_status (all are approved - no variation)
df_reg = df_reg.drop(['id', 'loan_approval_status'], axis=1)
# Display regression dataset shape
print("Regression dataset shape:", df_reg.shape)
# Display feature names
print("Features:", list(df_reg.columns))
# Summary statistics for regression dataset
df_reg.describe().transpose()

In [ ]:
# Save the regression dataset to CSV for use in Notebook 3
df_reg.to_csv('/content/drive/MyDrive/loan_regression_data.csv', index=False)
print("Saved: loan_regression_data.csv")
print(f"\nSummary:")
print(f"  Classification dataset: {df_class.shape}")
print(f"  Regression dataset:     {df_reg.shape}")